# Governance Email Generator

Reads `cleanup_tracker`, `workspace_inventory_snapshot`, and `governance_config` to generate Xebia-branded HTML emails and write them to `email_outbox`.

**No API calls. No admin permission. Reads and writes Delta tables only.**

### Emails generated
| Type | Recipient | When |
|------|-----------|------|
| Admin dashboard | Governance admin | Every run |
| 1st notice (friendly) | Item owner | New items entering warning cycle |
| 2nd warning (firm) | Item owner | Items escalated to warning_2 |
| Final warning (urgent) | Item owner | Items escalated to warning_3 |
| Deletion confirmation | Item owner | Items that were auto-deleted |


## 1. Setup

In [ ]:
import pandas as pd
import uuid
import logging
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("email_generator")

pipeline_run_id = str(uuid.uuid4())
run_timestamp   = datetime.now(timezone.utc).isoformat()
run_date_str    = datetime.now(timezone.utc).strftime("%B %d, %Y")

log.info(f"Pipeline run ID: {pipeline_run_id}")
log.info(f"Run date: {run_date_str}")


## 2. Load data

In [ ]:
# Config
df_cfg = spark.sql("SELECT config_key, config_value FROM governance_config").toPandas()
config = dict(zip(df_cfg["config_key"], df_cfg["config_value"]))
admin_email = config.get("admin_email", "<ADMIN_EMAIL>")
logo_url    = config.get("logo_url", "")
score_threshold = int(config.get("cleanup_score_threshold", "30"))

# Latest snapshot
df_snap = spark.sql("""
    SELECT * FROM workspace_inventory_snapshot
    WHERE snapshot_id = (
        SELECT snapshot_id FROM workspace_inventory_snapshot
        ORDER BY snapshot_time_utc DESC LIMIT 1
    )
""").toPandas()
for col in ["cleanup_candidate_score","is_stale","is_unused_artifact","has_missing_owner",
            "is_orphaned_model","is_orphaned_endpoint","is_duplicate_name",
            "days_since_modified","days_since_last_used","access_count_30d"]:
    if col in df_snap.columns:
        df_snap[col] = pd.to_numeric(df_snap[col], errors="coerce")

ws_names = df_snap["workspace_name"].dropna().unique().tolist() if not df_snap.empty else []
multi_ws = len(ws_names) > 1
workspace_name = (ws_names[0] if len(ws_names) == 1
                  else f"{len(ws_names)} workspaces" if ws_names
                  else "Fabric Workspace")

# workspace_id -> workspace_name, for labelling individual items
ws_lookup = (dict(zip(df_snap["workspace_id"].astype(str), df_snap["workspace_name"]))
             if not df_snap.empty else {})

log.info(f"Workspaces in snapshot: {len(ws_names)} — {ws_names}")

# Tracker
df_tracker = spark.sql("SELECT * FROM cleanup_tracker").toPandas()
for col in ["warning_count", "cleanup_score"]:
    if col in df_tracker.columns:
        df_tracker[col] = pd.to_numeric(df_tracker[col], errors="coerce")

# Latest audit entries (this run)
df_audit = spark.sql("""
    SELECT * FROM cleanup_audit_log
    ORDER BY timestamp DESC
""").toPandas()

log.info(f"Snapshot: {len(df_snap)} items | Tracker: {len(df_tracker)} items | Audit: {len(df_audit)} entries")
log.info(f"Admin email: {admin_email}")
log.info(f"Workspace: {workspace_name}")


## 3. Xebia HTML email base styles

In [ ]:
# ── Xebia brand colors ─────────────────────────────────
XEBIA_PURPLE    = "#6a1b6a"
XEBIA_PURPLE_LT = "#f3e8f3"
XEBIA_PURPLE_DK = "#4a124a"
WHITE           = "#ffffff"
GRAY_BG         = "#f7f7f7"
GRAY_BORDER     = "#e0e0e0"
GRAY_TEXT       = "#666666"
BLACK_TEXT       = "#333333"
RED_ACCENT      = "#d32f2f"
RED_BG          = "#fdecea"
AMBER_ACCENT    = "#f57c00"
AMBER_BG        = "#fff3e0"
GREEN_ACCENT    = "#2e7d32"

def email_header(title, subtitle="", urgent=False):
    """Xebia branded email header — white bg with purple accent bar."""
    accent = RED_ACCENT if urgent else XEBIA_PURPLE
    logo_html = (f'<img src="{logo_url}" alt="Xebia" width="130" style="display:block;border:0;" />'
                 if logo_url else
                 f'<span style="font-size:26px;font-weight:700;color:{XEBIA_PURPLE};letter-spacing:0.5px;">Xebia</span>')

    return f"""
    <table width="100%" cellpadding="0" cellspacing="0" border="0" style="max-width:680px;margin:0 auto;font-family:Segoe UI,Arial,sans-serif;background:{WHITE};">
    <tr><td style="height:5px;background:{accent};font-size:0;line-height:0;">&nbsp;</td></tr>
    <tr><td style="padding:20px 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
        <table width="100%" cellpadding="0" cellspacing="0" border="0">
        <tr>
            <td align="left" valign="middle">{logo_html}</td>
            <td align="right" valign="middle" style="font-size:11px;color:{GRAY_TEXT};letter-spacing:0.5px;text-transform:uppercase;">Fabric Workspace Governance</td>
        </tr>
        </table>
    </td></tr>
    <tr><td style="padding:0 28px 20px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};border-bottom:1px solid #f0f0f0;">
        <h1 style="margin:0 0 4px;font-size:20px;font-weight:600;color:{BLACK_TEXT};">{title}</h1>
        <p style="margin:0;font-size:13px;color:{GRAY_TEXT};">{subtitle}</p>
    </td></tr>
    """

def email_footer():
    return f"""
    <tr><td style="background:{GRAY_BG};padding:16px 28px;border-radius:0 0 8px 8px;border:1px solid {GRAY_BORDER};border-top:none;text-align:center;">
        <p style="margin:0;font-size:12px;color:{GRAY_TEXT};">
            Xebia &bull; Fabric Workspace Governance &bull; Automated report
        </p>
    </td></tr>
    </table>
    """

def kpi_card(label, value, color=BLACK_TEXT):
    return f"""
    <td style="background:{GRAY_BG};border-radius:6px;padding:12px 16px;text-align:center;width:25%;">
        <div style="font-size:24px;font-weight:700;color:{color};">{value}</div>
        <div style="font-size:11px;color:{GRAY_TEXT};margin-top:2px;">{label}</div>
    </td>
    """

def items_table(items_df, columns, headers, warning_level=None):
    """HTML table — text left-aligned, numbers centered."""
    header_bg = RED_BG if warning_level == 3 else AMBER_BG if warning_level == 2 else XEBIA_PURPLE_LT
    header_color = RED_ACCENT if warning_level == 3 else AMBER_ACCENT if warning_level == 2 else XEBIA_PURPLE_DK

    # Columns that should be center-aligned (numeric/short values)
    center_cols = {"Score", "cleanup_candidate_score", "cleanup_score", "score",
                   "Last modified", "_dsm", "Status", "_status", "status",
                   "Total flagged", "total", "Open warnings", "open_warnings",
                   "Resolved", "resolved", "Type", "type", "item_type"}

    col_widths = {
        "Name": "26%", "name": "26%", "item_name": "26%",
        "Type": "16%", "type": "16%", "item_type": "16%",
        "Owner": "24%", "created_by": "24%", "owner_email": "24%",
        "Score": "9%", "cleanup_candidate_score": "9%", "cleanup_score": "9%",
        "Status": "11%", "_status": "11%",
        "Reason": "22%", "_reasons": "22%",
        "Last modified": "13%", "_dsm": "13%",
        "Warnings sent": "24%",
        "Total flagged": "13%", "Open warnings": "14%", "Resolved": "12%",
    }

    html = (f'<table width="100%" cellpadding="0" cellspacing="0" border="0" '
            f'style="border:1px solid {GRAY_BORDER};border-radius:6px;border-collapse:collapse;'
            f'font-size:13px;table-layout:fixed;">')

    # Header
    html += '<tr>'
    for h, col in zip(headers, columns):
        w = col_widths.get(h, col_widths.get(col, "auto"))
        align = "center" if h in center_cols or col in center_cols else "left"
        html += (f'<th align="{align}" style="background:{header_bg};color:{header_color};'
                 f'padding:9px 10px;text-align:{align};font-weight:600;font-size:12px;'
                 f'border-bottom:1px solid {GRAY_BORDER};width:{w};">{h}</th>')
    html += '</tr>'

    # Rows
    for i, (_, row) in enumerate(items_df.iterrows()):
        bg = WHITE if i % 2 == 0 else GRAY_BG
        html += '<tr>'
        for h, col in zip(headers, columns):
            val = row.get(col, "")
            if pd.isna(val):
                val = "—"
            elif isinstance(val, float):
                val = int(val)
            val = str(val)
            display_val = val if len(val) <= 45 else val[:42] + "..."
            align = "center" if h in center_cols or col in center_cols else "left"
            html += (f'<td align="{align}" style="background:{bg};padding:9px 10px;'
                     f'border-bottom:1px solid {GRAY_BORDER};color:{BLACK_TEXT};'
                     f'text-align:{align};vertical-align:middle;word-break:break-word;" '
                     f'title="{val}">{display_val}</td>')
        html += '</tr>'

    html += '</table>'
    return html

def warning_badge(level):
    if level == 1:
        return f'<span style="background:{XEBIA_PURPLE_LT};color:{XEBIA_PURPLE};padding:2px 8px;border-radius:4px;font-size:11px;font-weight:600;">Notice 1 of 3</span>'
    elif level == 2:
        return f'<span style="background:{AMBER_BG};color:{AMBER_ACCENT};padding:2px 8px;border-radius:4px;font-size:11px;font-weight:600;">Warning 2 of 3</span>'
    elif level == 3:
        return f'<span style="background:{RED_BG};color:{RED_ACCENT};padding:2px 8px;border-radius:4px;font-size:11px;font-weight:600;">FINAL WARNING 3 of 3</span>'
    return ""

log.info("Email template functions loaded.")


## 4. Build admin dashboard email

In [ ]:
def build_admin_dashboard():
    total   = len(df_snap)
    stale   = int(df_snap["is_stale"].sum()) if "is_stale" in df_snap.columns else 0
    high    = int((df_snap["cleanup_candidate_score"] >= 50).sum())
    
    # Correctly count actively used — exclude "None" strings and actual nulls
    if "last_used_date" in df_snap.columns:
        active = int(df_snap["last_used_date"].apply(
            lambda x: pd.notna(x) and str(x).strip() not in ("", "None", "nan", "null")
        ).sum())
    else:
        active = 0
    
    unused  = int(df_snap["is_unused_artifact"].sum()) if "is_unused_artifact" in df_snap.columns else 0
    orphan_m = int(df_snap["is_orphaned_model"].sum()) if "is_orphaned_model" in df_snap.columns else 0
    orphan_e = int(df_snap["is_orphaned_endpoint"].sum()) if "is_orphaned_endpoint" in df_snap.columns else 0
    stale_pct = round(stale * 100 / total, 1) if total > 0 else 0

    # Usage breakdown — same "None" string handling
    def safe_dsm(x):
        if pd.isna(x) or str(x).strip() in ("", "None", "nan", "null"):
            return 9999
        return float(x)
    
    dsm_vals = df_snap["days_since_modified"].apply(safe_dsm) if "days_since_modified" in df_snap.columns else pd.Series([9999]*total)
    dormant = int((dsm_vals > 180).sum())
    aging   = int(((dsm_vals > 90) & (dsm_vals <= 180)).sum())
    recent  = total - dormant - aging - active

    # Tracker stats
    tracker_active = df_tracker[df_tracker["status"].isin(["new","warning_1","warning_2","warning_3"])] if not df_tracker.empty else pd.DataFrame()
    w1 = int((tracker_active["status"] == "warning_1").sum()) if not tracker_active.empty else 0
    w2 = int((tracker_active["status"] == "warning_2").sum()) if not tracker_active.empty else 0
    w3 = int((tracker_active["status"] == "warning_3").sum()) if not tracker_active.empty else 0
    new_count = int((tracker_active["status"] == "new").sum()) if not tracker_active.empty else 0
    resolved  = int((df_tracker["status"] == "resolved").sum()) if not df_tracker.empty else 0
    del_ready = int((df_tracker["status"].isin(["deletion_ready","pending_deletion"])).sum()) if not df_tracker.empty else 0

    # Audit: this run actions
    latest_run = df_audit["pipeline_run_id"].iloc[0] if not df_audit.empty else ""
    run_audit = df_audit[df_audit["pipeline_run_id"] == latest_run] if latest_run else pd.DataFrame()
    new_warnings = int(run_audit["action"].str.contains("warning_.*_sent").sum()) if not run_audit.empty else 0
    new_resolved = int((run_audit["action"] == "owner_resolved").sum()) if not run_audit.empty else 0
    new_flagged  = int((run_audit["action"] == "candidate_flagged").sum()) if not run_audit.empty else 0

    # Usage bar widths
    d_pct = round(dormant * 100 / total) if total else 0
    a_pct = round(aging * 100 / total) if total else 0
    r_pct = round(recent * 100 / total) if total else 0
    u_pct = 100 - d_pct - a_pct - r_pct

    # Build HTML
    html = email_header("Workspace Governance Report", f"{workspace_name} &bull; {run_date_str}")

    # KPI cards
    html += f"""
    <tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
        <table width="100%" cellpadding="0" cellspacing="8"><tr>
            {kpi_card("Total items", total)}
            {kpi_card("Stale items", f"{stale} ({stale_pct}%)", RED_ACCENT)}
            {kpi_card("High risk", high, RED_ACCENT)}
            {kpi_card("Actively used", active, GREEN_ACCENT)}
        </tr></table>
    </td></tr>
    """

    # Usage breakdown bar
    html += f"""
    <tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
        <p style="margin:0 0 6px;font-size:13px;font-weight:600;color:{BLACK_TEXT};">Usage breakdown</p>
        <div style="display:flex;height:20px;border-radius:4px;overflow:hidden;font-size:0;">
            <div style="width:{d_pct}%;background:{RED_ACCENT};"></div>
            <div style="width:{a_pct}%;background:{AMBER_ACCENT};"></div>
            <div style="width:{r_pct}%;background:#1976d2;"></div>
            <div style="width:{u_pct}%;background:{GREEN_ACCENT};"></div>
        </div>
        <table width="100%" cellpadding="0" cellspacing="0" style="margin-top:6px;font-size:11px;color:{GRAY_TEXT};">
        <tr>
            <td><span style="color:{RED_ACCENT};">&#9632;</span> Dormant &gt;180d ({dormant})</td>
            <td><span style="color:{AMBER_ACCENT};">&#9632;</span> Aging 90-180d ({aging})</td>
            <td><span style="color:#1976d2;">&#9632;</span> Recent ({recent})</td>
            <td><span style="color:{GREEN_ACCENT};">&#9632;</span> Active ({active})</td>
        </tr></table>
    </td></tr>
    """

    # This run's actions
    html += f"""
    <tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
        <p style="margin:0 0 8px;font-size:13px;font-weight:600;color:{BLACK_TEXT};">This run</p>
        <table cellpadding="0" cellspacing="0" style="font-size:13px;color:{BLACK_TEXT};">
            <tr><td style="padding:2px 16px 2px 0;">{new_flagged} new items flagged</td>
                <td style="padding:2px 16px 2px 0;">{new_warnings} warnings escalated</td></tr>
            <tr><td style="padding:2px 16px 2px 0;">{new_resolved} items resolved by owners</td>
                <td style="padding:2px 16px 2px 0;">{del_ready} items pending deletion</td></tr>
        </table>
    </td></tr>
    """

    # Warning pipeline summary
    html += f"""
    <tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
        <p style="margin:0 0 8px;font-size:13px;font-weight:600;color:{BLACK_TEXT};">Warning pipeline</p>
        <table width="100%" cellpadding="0" cellspacing="0" style="border:1px solid {GRAY_BORDER};border-radius:6px;border-collapse:separate;font-size:13px;">
            <tr><td style="background:{XEBIA_PURPLE_LT};padding:8px 16px;text-align:center;border-right:1px solid {GRAY_BORDER};">
                <div style="font-size:20px;font-weight:700;color:{XEBIA_PURPLE};">{new_count}</div>
                <div style="font-size:11px;color:{XEBIA_PURPLE};">New</div>
            </td>
            <td style="background:{XEBIA_PURPLE_LT};padding:8px 16px;text-align:center;border-right:1px solid {GRAY_BORDER};">
                <div style="font-size:20px;font-weight:700;color:{XEBIA_PURPLE};">{w1}</div>
                <div style="font-size:11px;color:{XEBIA_PURPLE};">Warning 1</div>
            </td>
            <td style="background:{AMBER_BG};padding:8px 16px;text-align:center;border-right:1px solid {GRAY_BORDER};">
                <div style="font-size:20px;font-weight:700;color:{AMBER_ACCENT};">{w2}</div>
                <div style="font-size:11px;color:{AMBER_ACCENT};">Warning 2</div>
            </td>
            <td style="background:{RED_BG};padding:8px 16px;text-align:center;border-right:1px solid {GRAY_BORDER};">
                <div style="font-size:20px;font-weight:700;color:{RED_ACCENT};">{w3}</div>
                <div style="font-size:11px;color:{RED_ACCENT};">Final</div>
            </td>
            <td style="background:{GRAY_BG};padding:8px 16px;text-align:center;">
                <div style="font-size:20px;font-weight:700;color:{GREEN_ACCENT};">{resolved}</div>
                <div style="font-size:11px;color:{GREEN_ACCENT};">Resolved</div>
            </td></tr>
        </table>
    </td></tr>
    """

    # Top cleanup candidates table
    top = df_snap.nlargest(15, "cleanup_candidate_score")
    if not top.empty:
        # Add warning status from tracker
        top = top.copy()
        top["_status"] = top["id"].apply(lambda x: df_tracker.loc[df_tracker["item_id"]==str(x), "status"].values[0] if not df_tracker.empty and str(x) in df_tracker["item_id"].values else "—")
        
        html += f"""
        <tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
            <p style="margin:0 0 8px;font-size:13px;font-weight:600;color:{BLACK_TEXT};">Top cleanup candidates</p>
            {items_table(top, ["name","type","created_by","cleanup_candidate_score","_status"],
                         ["Name","Type","Owner","Score","Status"])}
        </td></tr>
        """

    # Owner compliance table
    if not df_tracker.empty:
        owners = df_tracker[df_tracker["owner_email"].notna() & (df_tracker["owner_email"] != "") & (df_tracker["owner_email"] != "nan")]
        if not owners.empty:
            owner_stats = owners.groupby("owner_email").agg(
                total=("item_id", "count"),
                open_warnings=("status", lambda x: x.isin(["new","warning_1","warning_2","warning_3"]).sum()),
                resolved=("status", lambda x: (x == "resolved").sum()),
            ).reset_index().sort_values("open_warnings", ascending=False)

            html += f"""
            <tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
                <p style="margin:0 0 8px;font-size:13px;font-weight:600;color:{BLACK_TEXT};">Owner compliance</p>
                {items_table(owner_stats, ["owner_email","total","open_warnings","resolved"],
                             ["Owner","Total flagged","Open warnings","Resolved"])}
            </td></tr>
            """

    html += email_footer()
    return html

admin_html = build_admin_dashboard()
log.info(f"Admin dashboard email built: {len(admin_html)} chars")


## 5. Build owner warning emails

In [ ]:
def get_first_name(email):
    if not email or pd.isna(email) or email in ("", "nan", "None"):
        return "Team member"
    name = email.split("@")[0].replace(".", " ").split()
    return name[0].title() if name else "Team member"

def build_flag_reasons(item_id):
    """Build a comma-separated reason string from governance flags."""
    snap_row = df_snap[df_snap["id"].astype(str) == str(item_id)]
    if snap_row.empty:
        return "—"
    s = snap_row.iloc[0]
    reasons = []
    if s.get("is_stale") == 1: reasons.append("Stale")
    if s.get("is_unused_artifact") == 1: reasons.append("Unused")
    if s.get("is_orphaned_model") == 1: reasons.append("Orphaned model")
    if s.get("is_orphaned_endpoint") == 1: reasons.append("Orphaned endpoint")
    if s.get("has_missing_owner") == 1: reasons.append("No owner")
    if s.get("is_duplicate_name") == 1: reasons.append("Duplicate")
    return ", ".join(reasons) if reasons else "Score-based"

def build_owner_email(owner_email, items, warning_level, is_orphan=False):
    first_name = get_first_name(owner_email)
    n = len(items)
    if is_orphan:
        first_name = "Administrator"
    
    # Subject line
    if warning_level == 1:
        subject = f"Action needed: {n} workspace items require your review"
    elif warning_level == 2:
        subject = f"Reminder: {n} items still pending cleanup (2nd notice)"
    else:
        next_date = datetime.now(timezone.utc).strftime("%B %d, %Y")
        subject = f"FINAL NOTICE: {n} items will be auto-deleted on {next_date}"

    if is_orphan:
        subject = f"[NO OWNER] {subject}"
    
    urgent = warning_level == 3
    html = email_header(
        f"{'FINAL NOTICE' if urgent else 'Action needed'}: {n} items require review",
        f"{workspace_name} &bull; {run_date_str}",
        urgent=urgent
    )
    
    # Greeting + context
    if warning_level == 1:
        tone = f"""
        <p style="margin:0 0 12px;font-size:14px;color:{BLACK_TEXT};">Hi {first_name},</p>
        <p style="margin:0 0 12px;font-size:14px;color:{BLACK_TEXT};">
            Our automated workspace governance scan has identified <strong>{n} items</strong>
            you own in the <strong>{workspace_name}</strong> workspace that appear to be inactive or unused.
        </p>
        <p style="margin:0 0 12px;font-size:14px;color:{BLACK_TEXT};">
            Please review the items below and take one of these actions within the next 7 days:
        </p>
        <ul style="margin:0 0 16px;padding-left:20px;font-size:14px;color:{BLACK_TEXT};">
            <li><strong>Keep it</strong> &mdash; modify or use the item</li>
            <li><strong>Delete it</strong> &mdash; remove it from the workspace</li>
            <li><strong>Exempt it</strong> &mdash; reply to this email explaining why it should be retained</li>
        </ul>
        """
    elif warning_level == 2:
        tone = f"""
        <p style="margin:0 0 12px;font-size:14px;color:{BLACK_TEXT};">Hi {first_name},</p>
        <p style="margin:0 0 12px;font-size:14px;color:{AMBER_ACCENT};font-weight:600;">
            This is your second notice. {warning_badge(2)}
        </p>
        <p style="margin:0 0 12px;font-size:14px;color:{BLACK_TEXT};">
            The following <strong>{n} items</strong> in <strong>{workspace_name}</strong>
            were flagged in our previous scan and still require your action.
        </p>
        <p style="margin:0 0 16px;font-size:14px;color:{BLACK_TEXT};background:{AMBER_BG};padding:10px 14px;border-radius:6px;border-left:4px solid {AMBER_ACCENT};">
            Items not actioned after the next reminder will be <strong>automatically deleted</strong>.
        </p>
        """
    else:
        tone = f"""
        <p style="margin:0 0 12px;font-size:14px;color:{BLACK_TEXT};">Hi {first_name},</p>
        <p style="margin:0 0 12px;font-size:14px;color:{RED_ACCENT};font-weight:700;">
            This is your final notice. {warning_badge(3)}
        </p>
        <p style="margin:0 0 16px;font-size:14px;color:{BLACK_TEXT};background:{RED_BG};padding:10px 14px;border-radius:6px;border-left:4px solid {RED_ACCENT};">
            The following <strong>{n} items</strong> will be <strong>permanently deleted</strong>
            on the next pipeline run unless you take action now.
        </p>
        """

    html += f'<tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">{tone}'

    if is_orphan:
        html += f"""
        <p style="margin:0 0 14px;font-size:13px;color:{BLACK_TEXT};background:{AMBER_BG};
                  padding:10px 14px;border-radius:6px;border-left:4px solid {AMBER_ACCENT};">
            <strong>These items have no registered owner.</strong> They were routed to you as
            workspace administrator. They are usually system-generated, imported, or created by
            an account that has since been removed. They will follow the same escalation cycle,
            so please review or exempt them before deletion.
        </p>
        """

    # Build items table with reasons
    items_for_table = items.copy()
    items_for_table["_reasons"] = items_for_table["item_id"].apply(build_flag_reasons)
    items_for_table["_dsm"] = items_for_table["item_id"].apply(
        lambda x: df_snap.loc[df_snap["id"].astype(str)==str(x), "days_since_modified"].values[0]
        if str(x) in df_snap["id"].astype(str).values else "—"
    )
    items_for_table["_dsm"] = items_for_table["_dsm"].apply(
        lambda x: f"{int(float(x))}d" if pd.notna(x) and str(x) != "—" else "—"
    )

    if multi_ws:
        items_for_table["_ws"] = (items_for_table["workspace_id"].astype(str)
                                  .map(ws_lookup).fillna("—"))
        cols_    = ["item_name", "_ws", "item_type", "_dsm", "cleanup_score", "_reasons"]
        headers_ = ["Name", "Workspace", "Type", "Last modified", "Score", "Reason"]
    else:
        cols_    = ["item_name", "item_type", "_dsm", "cleanup_score", "_reasons"]
        headers_ = ["Name", "Type", "Last modified", "Score", "Reason"]

    html += items_table(items_for_table, cols_, headers_, warning_level=warning_level)

    # Footer context
    if warning_level == 1:
        html += f"""
        <p style="margin:16px 0 0;font-size:13px;color:{GRAY_TEXT};background:{GRAY_BG};padding:10px 14px;border-radius:6px;">
            <strong>What happens next?</strong> If no action is taken, you will receive a follow-up reminder.
            After 3 reminders, items will be automatically removed from the workspace.
        </p>
        """
    elif warning_level == 2:
        html += f"""
        <p style="margin:16px 0 0;font-size:13px;color:{GRAY_TEXT};">
            This is warning 2 of 3. One more reminder will be sent before automatic deletion.
        </p>
        """
    else:
        html += f"""
        <p style="margin:16px 0 0;font-size:13px;color:{RED_ACCENT};font-weight:600;">
            No further warnings will be sent. Items listed above will be permanently removed.
        </p>
        """

    html += '</td></tr>'
    html += email_footer()
    
    return subject, html

log.info("Owner email template function loaded.")


## 6. Build deletion confirmation emails

In [ ]:
def build_deletion_email(owner_email, items):
    first_name = get_first_name(owner_email)
    n = len(items)
    subject = f"{n} workspace items have been removed"
    
    html = email_header(
        f"{n} items removed from workspace",
        f"{workspace_name} &bull; {run_date_str}"
    )
    
    html += f"""
    <tr><td style="background:{WHITE};padding:0 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
        <p style="margin:0 0 12px;font-size:14px;color:{BLACK_TEXT};">Hi {first_name},</p>
        <p style="margin:0 0 16px;font-size:14px;color:{BLACK_TEXT};">
            The following items were automatically removed from the <strong>{workspace_name}</strong>
            workspace after 3 cleanup notices with no action taken.
        </p>
    """

    html += items_table(
        items,
        ["item_name", "item_type", "cleanup_score", "warning_1_date", "warning_2_date", "warning_3_date"],
        ["Name", "Type", "Score", "Warning 1", "Warning 2", "Warning 3"]
    )

    html += f"""
        <p style="margin:16px 0 0;font-size:13px;color:{GRAY_TEXT};background:{GRAY_BG};padding:10px 14px;border-radius:6px;">
            If you believe any item was removed in error, please contact the workspace administrator within 48 hours.
        </p>
    </td></tr>
    """
    
    html += email_footer()
    return subject, html

log.info("Deletion confirmation template function loaded.")


## 7. Generate all emails and write to outbox

In [ ]:
outbox_rows = []

# ── 7a. Admin dashboard email ──────────────────────────
outbox_rows.append({
    "email_id": str(uuid.uuid4()),
    "pipeline_run_id": pipeline_run_id,
    "email_type": "admin_report",
    "recipient": admin_email,
    "subject": f"Fabric Governance Report — {workspace_name} — {run_date_str}",
    "body_html": admin_html,
    "status": "pending",
    "created_at": run_timestamp,
    "workspace_id": df_snap["workspace_id"].iloc[0] if not df_snap.empty else "",
})
log.info(f"1 admin dashboard email queued for {admin_email}")


# ── Helpers ────────────────────────────────────────────
default_ws = df_snap["workspace_id"].iloc[0] if not df_snap.empty else ""

def is_blank_owner(series):
    """True where owner_email is null/empty/'nan'/'none'."""
    return (series.isna()
            | series.astype(str).str.strip().str.lower().isin(["", "nan", "none"]))

def queue_email(recipient, subject, html, email_type):
    outbox_rows.append({
        "email_id": str(uuid.uuid4()),
        "pipeline_run_id": pipeline_run_id,
        "email_type": email_type,
        "recipient": str(recipient),
        "subject": subject,
        "body_html": html,
        "status": "pending",
        "created_at": run_timestamp,
        "workspace_id": default_ws,
    })


# ── 7b. Owner warning emails ──────────────────────────
active_statuses = {"new", "warning_1", "warning_2", "warning_3"}
if not df_tracker.empty:
    active_items = df_tracker[df_tracker["status"].isin(active_statuses)].copy()
else:
    active_items = pd.DataFrame()

owner_email_count  = 0
orphan_email_count = 0

# status -> warning level used for tone/subject. "new" reads as a 1st notice.
LEVEL_MAP = [("warning_3", 3), ("warning_2", 2), ("warning_1", 1), ("new", 1)]

if not active_items.empty:
    blank        = is_blank_owner(active_items["owner_email"])
    orphan_items = active_items[blank]
    owned_items  = active_items[~blank]

    log.info(f"Active items: {len(active_items)} "
             f"({len(owned_items)} owned, {len(orphan_items)} ownerless)")

    # ── real owners ────────────────────────────────────
    for owner, grp in owned_items.groupby("owner_email"):
        for status_key, wl in LEVEL_MAP:
            batch = grp[grp["status"] == status_key]
            if batch.empty:
                continue
            subject, html = build_owner_email(owner, batch, wl)
            queue_email(owner, subject, html, f"owner_warning_{wl}")
            owner_email_count += 1

    # ── ownerless items routed to the admin ────────────
    if not orphan_items.empty:
        log.info(f"Routing {len(orphan_items)} ownerless items to {admin_email}")
        for status_key, wl in LEVEL_MAP:
            batch = orphan_items[orphan_items["status"] == status_key]
            if batch.empty:
                continue
            subject, html = build_owner_email(admin_email, batch, wl, is_orphan=True)
            queue_email(admin_email, subject, html, f"orphan_warning_{wl}")
            orphan_email_count += 1

log.info(f"{owner_email_count} owner emails, {orphan_email_count} orphan emails queued")


# ── 7c. Deletion confirmation emails ──────────────────
del_items = (df_tracker[df_tracker["status"] == "deleted"]
             if not df_tracker.empty else pd.DataFrame())
del_email_count = 0

if not del_items.empty:
    blank_del   = is_blank_owner(del_items["owner_email"])
    owned_del   = del_items[~blank_del]
    orphan_del  = del_items[blank_del]

    for owner, batch in owned_del.groupby("owner_email"):
        subject, html = build_deletion_email(owner, batch)
        queue_email(owner, subject, html, "deletion_confirmation")
        del_email_count += 1

    if not orphan_del.empty:
        subject, html = build_deletion_email(admin_email, orphan_del)
        subject = f"[NO OWNER] {subject}"
        queue_email(admin_email, subject, html, "deletion_confirmation")
        del_email_count += 1
        log.info(f"{len(orphan_del)} ownerless deletions reported to {admin_email}")

log.info(f"{del_email_count} deletion confirmation emails queued")


# ── TEST MODE: redirect all recipients ────────────────
test_recipient = str(config.get("test_mode_recipient", "")).strip()

if test_recipient and test_recipient.lower() not in ("none", "nan"):
    log.warning("=" * 60)
    log.warning(f"TEST MODE ACTIVE — all emails redirected to {test_recipient}")
    log.warning("Blank 'test_mode_recipient' in governance_config to disable.")
    log.warning("=" * 60)
    for row in outbox_rows:
        original = row["recipient"]
        row["recipient"] = test_recipient
        row["subject"] = f"[TEST -> {original}] {row['subject']}"
        log.info(f"  {row['email_type']:24s} intended for {original}")
else:
    log.info("Test mode OFF — emails will go to real owners.")


# ── Write to email_outbox ─────────────────────────────
log.info(f"Total emails to write: {len(outbox_rows)}")

if outbox_rows:
    df_outbox = pd.DataFrame(outbox_rows)
    spark_outbox = spark.createDataFrame(df_outbox.astype(str))
    spark_outbox.write.format("delta").mode("append").saveAsTable("email_outbox")
    log.info(f"✔ {len(outbox_rows)} emails written to email_outbox")
else:
    log.info("No emails to write.")

## 8. Summary

In [ ]:
print("=" * 60)
print("  EMAIL GENERATOR — SUMMARY")
print("=" * 60)
print(f"  Pipeline run:   {pipeline_run_id}")
print(f"  Timestamp:      {run_date_str}")
print(f"  ")
print(f"  EMAILS GENERATED")
print(f"  ────────────────")
print(f"  Admin dashboard:          1  →  {admin_email}")
print(f"  Owner warnings:           {owner_email_count}")
print(f"  Deletion confirmations:   {del_email_count}")
print(f"  Total in outbox:          {len(outbox_rows)}")
print(f"  ")
print(f"  EMAIL TYPES BREAKDOWN")
print(f"  ─────────────────────")
if outbox_rows:
    df_ob = pd.DataFrame(outbox_rows)
    for etype, count in df_ob["email_type"].value_counts().items():
        print(f"  {etype:30s} {count}")
print("=" * 60)


## 9. Preview admin dashboard email

In [ ]:
# Render the admin dashboard email in the notebook for preview
displayHTML(admin_html)


## 10. Preview a sample owner email

In [ ]:
# Preview ALL owner warning emails
owner_emails = [r for r in outbox_rows if r["email_type"].startswith("owner_warning")]
if owner_emails:
    print(f"Total owner emails: {len(owner_emails)}\n")
    for i, sample in enumerate(owner_emails):
        print(f"{'='*60}")
        print(f"Email {i+1}: {sample['email_type']} → {sample['recipient']}")
        print(f"Subject: {sample['subject']}")
        print(f"{'='*60}")
        displayHTML(sample["body_html"])
else:
    print("No owner warning emails generated.")

## Next step

The `email_outbox` table now has all pending emails. The Fabric Data Pipeline reads them using a Lookup activity and sends via the Outlook Activity in a ForEach loop.

```
Pipeline Stage 4: Lookup → email_outbox WHERE email_type = 'admin_report'
                  → Outlook Activity (To, Subject, Body from Lookup output)

Pipeline Stage 5: Lookup → email_outbox WHERE email_type LIKE 'owner_warning%'
                  → ForEach → Outlook Activity
```
